# Intro

- In the previous module we evaluated our system offline, before anyone used it.
    - We measured search quality with Hit Rate and MRR,
    - and answer quality with cosine similarity and an LLM judge.

- But once we deploy, real people start asking real questions. The offline numbers stop telling the whole story.
    - We don't know how long answers take, what they cost, or whether anyone finds them useful.
    - We need to watch the system while it runs.

- That's monitoring: online evaluation.
    - We collect metrics from the running system and put them on a dashboard.
    - Then we can see how it performs with real traffic.

- For every question that comes in, there's a lot we can capture:
    - The instructions, prompt, and model behind the answer
    - Input and output tokens, and how much the call cost
    - Response time: how long the person waited
    - User feedback: a thumbs up or thumbs down on the answer
    - Relevance: does the answer address the question?

- Collecting and viewing all of this takes three new pieces:
    - a user interface where people ask questions
    - a database to store conversations and feedback
    - a dashboard to visualize the metrics

- We build all three on top of the RAG pipeline from the earlier modules.
    - We don't rebuild the RAG part. We wrap it in a Streamlit app and save every interaction to PostgreSQL.
    - Then we put a dashboard in front of the data. At the end we add Grafana for a more powerful view.

- We focus on **RAG** here. Monitoring an **agent** works almost the same way, so we leave it as **homework**.
    - The [agents module](../01_agentic_rag/) already has the pieces you need to apply these same ideas there.

# Assistant

- Before we monitor anything, we need something to monitor. So we start with a RAG pipeline that answers questions about our courses.
- We won't build it from scratch. We already did that in the earlier modules, and the flow is the same three steps as always.
    - First we search the FAQ for the questions most relevant to the user's question.
    - Then we build a prompt from that question plus the documents we found.
    - Finally we send it to the LLM, which gives us the answer.
- That's the whole pipeline, and we reuse it as-is.

## Creating the assistant

- [`assistant.py`](../src/assistant.py) loads the data and builds the index, then hands both to `RAGBase`.
- We pass our own instructions template here.
- Run the assistant

In [ ]:
%%bash
uv run python ../src/assistant.py

In [ ]:
%%bash
uv run python ../src/assistant.py "a different query"

We'll run this command again and again, and typing it in full every time gets old. So we put it in a [`Makefile`](../Makefile).

In [ ]:
%%bash
# sudo apt install make
make -C .. run ARGS='"a different query"'

# Chat App

- The command line works, but we want something closer to how a person would actually talk to the assistant.
- So we wrap it in a small web interface. We use **Streamlit**, a Python framework for building front ends with almost no code.
- This isn't the final product, and it isn't meant to be pretty.

In [ ]:
%%bash
uv add streamlit

- Create and Run [`app.py`](../src/app.py)
    ```bash
    uv run streamlit run ../src/app.py
    ```
- That's another long command, so it goes into the [`Makefile`](../Makefile) too.
- The RAG works, but right now we track nothing about it: no response time, no token usage, no cost.
- That's exactly the visibility monitoring is supposed to give us, so that's what we add next.

In [ ]:
%%bash
make chat # at project root and only terminal works not notebook